---
---
---

In [5]:
# ============================================
# video → bbox追従 → zoom in（サイズ維持）
# ============================================

import cv2
import torch
import numpy as np
from pathlib import Path
from PIL import Image

# ==============================
# 入力
# ==============================
mp4_path = "/workspace/data/videos/DaUJkmBvTKM_2_0to150.mp4"
out_path = "/workspace/notebook/output/background_change.mp4"
instruction = "zoom in on face"


In [6]:
import subprocess
import json

def get_video_info_ffprobe(path):
    cmd = [
        "ffprobe",
        "-v", "error",
        "-select_streams", "v:0",
        "-show_entries", "stream=r_frame_rate,avg_frame_rate",
        "-show_entries", "stream=width,height,r_frame_rate,avg_frame_rate",
        "-show_entries", "format=duration",
        "-of", "json",
        path
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)
    data = json.loads(result.stdout)

    # duration（秒）
    duration = float(data["format"]["duration"])

    # fps（avg_frame_rate優先）
    fps_str = data["streams"][0]["avg_frame_rate"]
    num, den = map(int, fps_str.split("/"))
    fps = num / den if den != 0 else 0

    # shapeを取得
    width = int(data["streams"][0]["width"])
    height = int(data["streams"][0]["height"])

    # dict で返す
    return {"duration": duration, "fps": fps, "width": width, "height": height}

In [7]:
# mp4情報の取得
video_info = get_video_info_ffprobe(mp4_path)

print("duration(sec):", video_info["duration"])
print("fps:", video_info["fps"])
print("frame_count:", int(video_info["duration"] * video_info["fps"]))
print(f"shape: {video_info['width']}x{video_info['height']}")

fps = video_info["fps"]
# fps = int(round(video_info["fps"]))
# print("fps rounded:", fps)

# ==============================
# 1. 動画読み込み
# 背景意図：
# ・フレーム単位処理のため全フレーム保持
# ・RGBに統一（DINO前提）
# ==============================
frames_rgb = []
cap = cv2.VideoCapture(mp4_path)

frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frames_rgb.append(frame_rgb)

cap.release()

if not frames_rgb:
    raise RuntimeError("Failed to read video")

h, w, _ = frames_rgb[0].shape
T_total = len(frames_rgb)


duration(sec): 5.0
fps: 25.0
frame_count: 125
shape: 1920x1080


In [8]:
# ==============================
# 2. GroundingDINOロード
# 背景意図：
# ・毎フレームで対象再検出（tracking無しで簡易対応）
# ==============================
from groundingdino.util.inference import load_model, predict
from groundingdino.datasets import transforms as T

CONFIG_PATH = "/workspace/third_party/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py"
CHECKPOINT = "/workspace/third_party/GroundingDINO/weights/groundingdino_swint_ogc.pth"

device = "cuda" if torch.cuda.is_available() else "cpu"
dino_model = load_model(CONFIG_PATH, CHECKPOINT, device=device)

TEXT_PROMPT = "face . person ."
BOX_TRESHOLD = 0.35
TEXT_TRESHOLD = 0.25


final text_encoder_type: bert-base-uncased


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15985.80it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
# ============================================
# video → stable zoom（完全修正版）
# ============================================

import cv2
import torch
import numpy as np
from PIL import Image

# ==============================
# 入力
# ==============================
zoom_factor = 1.0  # ← ユーザー指定


# ==============================
# 2. DINO
# ==============================
from groundingdino.util.inference import load_model, predict
from groundingdino.datasets import transforms as T

CONFIG_PATH = "/workspace/third_party/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py"
CHECKPOINT = "/workspace/third_party/GroundingDINO/weights/groundingdino_swint_ogc.pth"

device = "cuda" if torch.cuda.is_available() else "cpu"
model = load_model(CONFIG_PATH, CHECKPOINT, device=device)

def transform_image(image):
    transform = T.Compose([
        T.RandomResize([800], max_size=1333),
        T.ToTensor(),
        T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
    return transform(Image.fromarray(image), None)[0]

def box_cxcywh_to_xyxy(boxes):
    cx, cy, bw, bh = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    x1 = cx - bw / 2
    y1 = cy - bh / 2
    x2 = cx + bw / 2
    y2 = cy + bh / 2
    return torch.stack([x1, y1, x2, y2], dim=1)

# ==============================
# 3. 初回bbox取得（←ここ重要）
# 背景意図：
# ・ズームスケールは1フレーム目で固定
# ==============================
img0 = frames_rgb[0]
tensor0 = transform_image(img0)

boxes, logits, phrases = predict(
    model=model,
    image=tensor0,
    caption="face . person .",
    box_threshold=0.35,
    text_threshold=0.25,
    device=device,
)

boxes = box_cxcywh_to_xyxy(boxes)
boxes = boxes * torch.tensor([w,h,w,h])
boxes = boxes.cpu().numpy()

# face優先
target_box = boxes[0]

x1,y1,x2,y2 = target_box
cx = (x1+x2)/2
cy = (y1+y2)/2
bw = (x2-x1)
bh = (y2-y1)

# ==============================
# 4. ズームスケジュール生成（固定）
# 背景意図：
# ・frameごとに同じスケール規則を使う
# ==============================

# bboxの長辺
bbox_long = max(bw, bh)

# 最終的に画面いっぱいになるスケール
target_scale = (bbox_long / max(w,h)) / zoom_factor

# t=0 → 1.0（全体）
# t=1 → target_scale
scales = np.linspace(1.0, target_scale, T_total)

# ==============================
# 5. 各フレーム処理
# ==============================
output_frames = []

for t, frame in enumerate(frames_rgb):

    scale = scales[t]

    # ---- 現在のcropサイズ（アスペクト比維持） ----
    crop_w = int(w * scale)
    crop_h = int(h * scale)

    # 中心固定
    x1 = int(cx - crop_w/2)
    y1 = int(cy - crop_h/2)
    x2 = int(cx + crop_w/2)
    y2 = int(cy + crop_h/2)

    # clamp
    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(w, x2)
    y2 = min(h, y2)

    cropped = frame[y1:y2, x1:x2]

    # resizeで元サイズへ戻す
    resized = cv2.resize(cropped, (w,h), interpolation=cv2.INTER_LINEAR)

    output_frames.append(resized)


final text_encoder_type: bert-base-uncased


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16947.89it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
# ==============================
# SAMロード
# ==============================
# SAM
import sys
from pathlib import Path
sys.path.append("/workspace/third_party/segment-anything")
from segment_anything import sam_model_registry, SamPredictor
from segment_anything import sam_model_registry, SamPredictor

# ==============================
# SAM設定
# ==============================
# SAM_CHECKPOINT = ROOT / "weights/sam_vit_h_4b8939.pth"
SAM_CHECKPOINT = Path("/workspace/weights/sam_vit_h_4b8939.pth")
SAM_TYPE = "vit_h"

device = "cuda" if torch.cuda.is_available() else "cpu"

# ==============================
# モデルロード
# ==============================
sam = sam_model_registry[SAM_TYPE](checkpoint=str(SAM_CHECKPOINT))
sam.to(device=device)
predictor = SamPredictor(sam)

In [ ]:
import subprocess
import os 
tmp_dir = "/workspace/notebook/output/tmp_frames"
os.makedirs(tmp_dir, exist_ok=True)
out_path_3 = out_path.replace(".mp4", "_v3.mp4")  # テスト用パス

# フレーム保存
for i, f in enumerate(output_frames):
    path = f"{tmp_dir}/{i:05d}.png"
    cv2.imwrite(path, cv2.cvtColor(f, cv2.COLOR_RGB2BGR))

# ffmpegで動画化
cmd = f"""
ffmpeg -y -framerate {fps} -i {tmp_dir}/%05d.png \
-c:v libx264 -pix_fmt yuv420p {out_path_3}
"""
subprocess.run(cmd, shell=True)

# mp4を表示する.
from IPython.display import Video
Video(out_path_3, embed=True, width=640, height=360)

In [ ]:
# output_frames　を 列数指定、行数は自動で調整して表示する.
import matplotlib.pyplot as plt
cols = 5
rows = (len(output_frames) + cols - 1) // cols
plt.figure(figsize=(20, 4 * rows))
for i, frame in enumerate(output_frames):
    plt.subplot(rows, cols, i + 1)
    plt.imshow(frame)
    plt.axis('off')
plt.tight_layout()

In [ ]:
# 各フレームの サイズの分布を表示する. width vs height
frame_sizes = [(f.shape[1], f.shape[0]) for f in output_frames]
widths, heights = zip(*frame_sizes)
min_width, max_width = min(widths), max(widths)
min_height, max_height = min(heights), max(heights)
print(f"Width: min={min_width}, max={max_width}")
print(f"Height: min={min_height}, max={max_height}")
plt.figure(figsize=(10,5))
plt.plot(widths, heights, 'o')
plt.xlabel('Width')
plt.ylabel('Height')
plt.title('Frame Size Distribution')
plt.grid(True)
plt.xlim(1919, 1921)
plt.ylim(1079, 1081)
plt.show()